In [1]:
import re
import numpy as np
import pandas as pd
import sklearn

from sklearn.model_selection import train_test_split, KFold
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

from pathlib import Path

In [2]:
print(np.__version__)
print(pd.__version__)
print(sklearn.__version__)

2.5.3
3.0.6
1.9.1


In [3]:
DATA_PATH = Path("../datasets").resolve()
datasets = list(DATA_PATH.glob("*.csv", case_sensitive=False))
datasets

[WindowsPath('D:/Users/Esperine/Workspace3/mlzc-main/datasets/data.csv'),
 WindowsPath('D:/Users/Esperine/Workspace3/mlzc-main/datasets/WA_Fn-UseC_-Telco-Customer-Churn.csv')]

In [4]:
df = pd.read_csv(datasets[1])
print(df.shape)
df.head()

(7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## Preliminaries

In [5]:
def normalize_name(s: str) -> str:
    # Insert underscore between consecutive capitals followed by lowercase 
    # (e.g., 'XMLHttp' -> 'XML_Http')
    s = re.sub(r"([A-Z]+)([A-Z][a-z])", r"\1_\2", s)
    # Insert underscore between lowercase/numbers and following uppercase 
    # (e.g., 'HttpReq' -> 'Http_Req')
    s = re.sub(r"([a-z\d])([A-Z])", r"\1_\2", s)

    return s.lower()

# Normalize column names
df.columns = df.rename(columns=normalize_name).columns

# Fix format in each categorical variables
categorical_columns = list(df.dtypes[df.dtypes == "str"].index)
for c in categorical_columns:
    df[c] = df[c].str.lower().str.replace(' ', '_')

# Fill NULLs with zero in `total_charges`
df["total_charges"] = pd.to_numeric(df["total_charges"], errors="coerce").fillna(0)

# Convert `churn` into binary values
df["churn"] = (df["churn"] == "yes").astype(int)

In [6]:
RANDOM_SEED = 1

# Train-val-test split
df_full_train, df_test = train_test_split(
    df, 
    test_size=0.2, 
    random_state=RANDOM_SEED
)
df_train, df_val = train_test_split(
    df_full_train, 
    test_size=0.25, 
    random_state=RANDOM_SEED
)

# Resetting indices
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)
# df_full_train = df_full_train.reset_index(drop=True)

# Extract target
y_train = df_train["churn"].values
y_val = df_val["churn"].values
y_test = df_test["churn"].values
# y_full_train = df_full_train["churn"].values

# Remove target from dataframes
df_train.drop(["churn", "customer_id"], axis=1, inplace=True)
df_val.drop(["churn", "customer_id"], axis=1, inplace=True)
df_test.drop(["churn", "customer_id"], axis=1, inplace=True)
# df_full_train.drop("churn", axis=1, inplace=True)

In [7]:
df.shape, df_train.shape, df_val.shape, df_test.shape, df_full_train.shape

((7043, 21), (4225, 19), (1409, 19), (1409, 19), (5634, 21))

In [8]:
numerical_features = ["tenure", "monthly_charges", "total_charges"]

exclude_cols = numerical_features[:]
exclude_cols.extend(["churn", "customer_id"])
categorical_features = df_full_train.columns.difference(exclude_cols, sort=False).to_list()

all_features = categorical_features + numerical_features

In [9]:
len(categorical_features), len(numerical_features), len(all_features)

(16, 3, 19)

In [10]:
def train(df_train, y_train, C=1.0):
    dicts = df_train[all_features].to_dict(orient="records")

    dv = DictVectorizer(sparse=False)
    X_train = dv.fit_transform(dicts)

    model = LogisticRegression(
        solver="liblinear", 
        C=C, 
        max_iter=1000, 
        random_state=RANDOM_SEED
    )
    model.fit(X_train, y_train)
    
    return dv, model


def predict(df, dv, model):
    dicts = df[all_features].to_dict(orient="records")

    X = dv.transform(dicts)
    y_pred = model.predict_proba(X)[:, 1]

    return y_pred

In [11]:
n_splits = 5

for C in [0.001, 0.01, 0.1, 0.5, 1, 5, 10]:
    kfold = KFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_SEED)

    scores = []

    for train_idx, val_idx in kfold.split(df_full_train):
        df_train = df_full_train.iloc[train_idx]
        df_val = df_full_train.iloc[val_idx]

        y_train = df_train["churn"].values
        y_val = df_val["churn"].values

        dv, model = train(df_train, y_train, C=C)
        y_pred = predict(df_val, dv, model)

        auc = roc_auc_score(y_val, y_pred)
        scores.append(auc)

    print('C=%s %.3f +- %.3f' % (C, np.mean(scores), np.std(scores)))

C=0.001 0.825 +- 0.013
C=0.01 0.839 +- 0.009
C=0.1 0.841 +- 0.007
C=0.5 0.841 +- 0.007
C=1 0.841 +- 0.007
C=5 0.841 +- 0.007
C=10 0.841 +- 0.007


In [12]:
scores

[0.8422141281566202,
 0.8452171070056945,
 0.8338350391330523,
 0.8322708160133018,
 0.8522167487684729]

In [13]:
df_full_train.shape

(5634, 21)

In [14]:
# Final Model

dv, model = train(df_full_train, df_full_train["churn"].values, C=1.0)
y_pred = predict(df_test, dv, model)

auc = roc_auc_score(y_test, y_pred)
auc

0.8579400803839363

In [15]:
model.classes_

array([0, 1])

## Sample

In [16]:
sample_customer = {
    'gender': 'female',
    'seniorcitizen': 0,
    'partner': 'yes',
    'dependents': 'no',
    'phoneservice': 'no',
    'multiplelines': 'no_phone_service',
    'internetservice': 'fiber_optic',
    'onlinesecurity': 'no',
    'onlinebackup': 'yes',
    'deviceprotection': 'no',
    'techsupport': 'no',
    'streamingtv': 'yes',
    'streamingmovies': 'no',
    'contract': 'one_year',
    'paperlessbilling': 'yes',
    'paymentmethod': 'electronic_check',
    'tenure': 36,
    'monthlycharges': 100.67,
    'totalcharges': 3080.5
}

In [17]:
X = dv.transform(sample_customer)
sample_churn = model.predict_proba(X)
sample_churn

array([[0.93522588, 0.06477412]])

## Export model

In [18]:
import pickle

In [19]:
with open("model.bin", "wb") as f_out:
    pickle.dump((dv, model), f_out)

In [20]:
!powershell ls



    Directory: D:\Users\Esperine\Workspace3\mlzc-main\04-deployment


Mode                 LastWriteTime         Length Name                                                                 
----                 -------------         ------ ----                                                                 
d-----        10/15/2026   1:39 AM                .ipynb_checkpoints                                                   
d-----        10/14/2026  10:12 PM                .venv                                                                
d-----        10/14/2026  10:10 PM                src                                                                  
-a----        10/14/2026  10:10 PM              5 .python-version                                                      
-a----        10/15/2026   3:37 AM         236053 churn-model.ipynb                                                    
-a----        10/15/2026  10:43 AM           2458 model.bin                             

## Load model

In [21]:
with open("model.bin", "rb") as f_in:
    dv, model = pickle.load(f_in)

In [22]:
dv

DictVectorizer(sparse=False)

In [23]:
model

,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",1
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multiclass` problems (`n_classes >= 3`), all solvers except 'liblinear' minimize the full multinomial loss, 'liblinear' will raise an error.- 'newton-cholesky' is a good choice for `n_samples` >> `n_features * n_classes`, especially with one-hot encoded categorical features with rare categories. Be aware that the memory usage of this solver has a quadratic dependency on `n_features * n_classes` because it explicitly computes the full Hessian matrix.- For small datasets, 'liblinear' is a good choice, whereas 'sag' and 'saga' are faster for large ones;- 'liblinear' can only handle binary classification by default. To apply a one-versus-rest scheme for the multiclass setting one can wrap it with the :class:`~sklearn.multiclass.OneVsRestClassifier`... warning:: The choice of the algorithm depends on the penalty chosen (`l1_ratio=0` for L2-penalty, `l1_ratio=1` for L1-penalty and `0 < l1_ratio < 1` for Elastic-Net) and on (multinomial) multiclass support: ================= ======================== ====================== solver l1_ratio multinomial multiclass ================= ======================== ====================== 'lbfgs' l1_ratio=0 yes 'liblinear' l1_ratio=1 or l1_ratio=0 no 'newton-cg' l1_ratio=0 yes 'newton-cholesky' l1_ratio=0 yes 'sag' l1_ratio=0 yes 'saga' 0<=l1_ratio<=1 yes ================= ======================== ======================.. note:: 'sag' and 'saga' fast convergence is only guaranteed on features with approximately the same scale. You can preprocess the data with a scaler from :mod:`sklearn.preprocessing`... seealso:: Refer to the :ref:`User Guide <Logistic_regression>` for more information regarding :class:`LogisticRegression` and more specifically the :ref:`Table <logistic_regression_solvers>` summarizing solver/penalty supports... versionadded:: 0.17 Stochastic Average Gradient (SAG) descent solver. Multinomial support in version 0.18... versionadded:: 0.19 SAGA solver... versionchanged:: 0.22 The default solver changed from 'liblinear' to 'lbfgs' in 0.22... versionadded:: 1.2 newton-cholesky solver. Multinomial support in version 1.6.",'liblinear'
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1

In [24]:
X = dv.transform(sample_customer)
sample_churn = model.predict_proba(X)
sample_churn

array([[0.93522588, 0.06477412]])

## Training pipeline

We can simplify the code by creating a pipeline for the training

In [26]:
from sklearn.pipeline import make_pipeline

In [27]:
churn_model = make_pipeline(
    DictVectorizer(sparse=False),
    LogisticRegression(
        solver="liblinear", 
        C=1.0, 
        max_iter=1000, 
        random_state=RANDOM_SEED
    )
)

In [28]:
train_dict = df_full_train[all_features].to_dict(orient="records")
y_full_train = df_full_train["churn"].values

churn_model.fit(train_dict, y_full_train)

# auc = roc_auc_score(y_test, y_pred)
# auc

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('dictvectorizer', ...), ('logisticregression', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
,"sparse sparse: bool, default=TrueWhether transform should produce scipy.sparse matrices.",False
,"dtype dtype: dtype, default=np.float64The type of feature values. Passed to Numpy array/scipy.sparse matrixconstructors as the dtype argument.",<class 'numpy.float64'>
,"separator separator: str, default=""=""Separator string used when constructing new features for one-hotcoding.",'='
,"sort sort: bool, default=TrueWhether ``feature_names_`` and ``vocabulary_`` should besorted when fitting.",True
Name,Type,Value


In [46]:
sample_churn = churn_model_pl.predict_proba(sample_customer)
sample_churn

array([[0.93522588, 0.06477412]])

In [30]:
test_dicts = df_test.to_dict(orient="records")

y_pred = churn_model.predict_proba(test_dicts)[:, 1]

auc = roc_auc_score(y_test, y_pred)
auc

0.8579400803839363

In [31]:
y_test

array([0, 0, 0, ..., 0, 0, 1], shape=(1409,))

In [32]:
with open("model.bin", "wb") as f_out:
    pickle.dump(churn_model, f_out)

In [36]:
with open("model.bin", "rb") as f_in:
    model = pickle.load(f_in)

In [38]:
model

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('dictvectorizer', ...), ('logisticregression', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
,"sparse sparse: bool, default=TrueWhether transform should produce scipy.sparse matrices.",False
,"dtype dtype: dtype, default=np.float64The type of feature values. Passed to Numpy array/scipy.sparse matrixconstructors as the dtype argument.",<class 'numpy.float64'>
,"separator separator: str, default=""=""Separator string used when constructing new features for one-hotcoding.",'='
,"sort sort: bool, default=TrueWhether ``feature_names_`` and ``vocabulary_`` should besorted when fitting.",True
Name,Type,Value


In [39]:
sample_churn = model.predict_proba(sample_customer)
sample_churn

array([[0.93522588, 0.06477412]])